In [ ]:
hold_count = 4
rebalance_days = 5
from bigquant import bigtrader, dai
import pandas as pd
import numpy as np
from bigtrader.finance.commission import PerOrder
from scipy.optimize import minimize

# 定义指定的 ETF 列表
specified_etfs = [
'511380.SH',     # 可转债 
'513520.SH',      # 日经
'159980.SZ',       # 有色
# '510050.SH',       # 上证50 
'160723.SZ',    # 原油 
'513100.SH',      # 纳指
'159985.SZ',     # 豆粕
'513500.SH',     # 标普
'518880.SH',     # 黄金
'513130.SH',     # 恒生科技
'510300.SH',    # 沪深300
'159915.SZ',    # 创业板
]   



def trend_score(close, period=25):
    """向量化计算趋势评分：年化收益率 × R平方"""
    if len(close) < period:
        return np.full_like(close, np.nan)

    y = np.log(close)
    windows = np.lib.stride_tricks.sliding_window_view(y, window_shape=period)
    x = np.arange(period)

    # 预计算固定值
    n = period
    sum_x = x.sum()
    sum_x2 = (x **2).sum()
    denominator = n * sum_x2 - sum_x** 2

    # 滑动窗口统计量
    sum_y = windows.sum(axis=1)
    sum_xy = (windows * x).sum(axis=1)

    # 回归系数
    slope = (n * sum_xy - sum_x * sum_y) / denominator
    intercept = (sum_y - slope * sum_x) / n

    # 年化收益率
    annualized_returns = np.exp(slope * 250) - 1

    # R平方计算
    y_pred = slope[:, None] * x + intercept[:, None]
    residuals = windows - y_pred
    ss_res = np.sum(residuals **2, axis=1)

    sum_y2 = np.sum(windows** 2, axis=1)
    ss_tot = sum_y2 - (sum_y **2) / n
    r_squared = 1 - (ss_res / ss_tot)
    r_squared = np.nan_to_num(r_squared, nan=0.0)  # 处理零方差情况

    # 综合评分
    score = annualized_returns * r_squared

    # 对齐原始序列长度
    full_score = pd.Series(index=close.index)
    full_score[period - 1:] = score
    return full_score

def calculate_returns(context_data, stock_targets, lookback_period=120):
    """计算指定股票的历史收益率"""
    # 获取最新交易日
    latest_date = context_data['date'].max()
    
    # 获取回溯期数据
    historical_data = context_data[
        (context_data['date'] <= latest_date) & 
        (context_data['instrument'].isin(stock_targets))
    ].sort_values(['instrument', 'date']).groupby('instrument').tail(lookback_period) 
    
    print('===最后数据日期=====:', latest_date, historical_data.date.max())

    
    # 计算收益率
    returns = historical_data.pivot(index='date', columns='instrument', values='close').pct_change().dropna()
    return returns

def mean_variance_optimization(returns, risk_aversion=1.0, min_weight=0.05, max_weight=0.3):

    expected_returns = returns.mean() * 252  # 年化预期收益
    cov_matrix = returns.cov() * 252  # 年化协方差矩阵
    n_assets = len(expected_returns)
    
    # 初始权重：等权分配（确保初始值满足所有约束）
    initial_weights = np.ones(n_assets) / n_assets
    # 强制初始权重满足上下限（避免优化起点违规）
    initial_weights = np.clip(initial_weights, min_weight, max_weight)
    initial_weights = initial_weights / initial_weights.sum()  # 重新归一化确保和为1

    # 目标函数：最大化风险调整后收益（负号用于minimize的最小化）
    def objective(weights):
        portfolio_return = np.sum(expected_returns * weights)
        portfolio_variance = np.dot(weights.T, np.dot(cov_matrix, weights))
        return -(portfolio_return - 0.5 * risk_aversion * portfolio_variance)
    
    # 约束条件
    constraints = [
        {'type': 'eq', 'fun': lambda x: np.sum(x) - 1},  # 总权重=1
    ]
    # 为每只ETF添加上下限约束（避免单只权重为0或超限）
    for i in range(n_assets):
        # 单只权重 ≤ max_weight
        constraints.append({'type': 'ineq', 'fun': lambda x, idx=i: max_weight - x[idx]})
        # 单只权重 ≥ min_weight（核心：避免0权重）
        constraints.append({'type': 'ineq', 'fun': lambda x, idx=i: x[idx] - min_weight})
    
    # 权重边界（双重保障，与约束条件一致）
    bounds = [(min_weight, max_weight) for _ in range(n_assets)]
    
    # 执行优化（增加迭代次数，确保收敛）
    result = minimize(
        objective,
        initial_weights,
        method='SLSQP',
        bounds=bounds,
        constraints=constraints,
        options={'maxiter': 5000, 'ftol': 1e-9, 'eps': 1e-8}
    )
    
    # 处理优化失败的极端情况（强制分配满足约束的权重）
    if not result.success:
        context.log.warning("优化未收敛，使用强制均分权重（满足上下限）")
        # 强制等权分配，确保满足上下限
        weights = np.ones(n_assets) / n_assets
        weights = np.clip(weights, min_weight, max_weight)
        weights = weights / weights.sum()
        return pd.Series(weights, index=expected_returns.index)
    
    # 最终权重再检查（确保满足所有约束）
    weights = result.x
    weights = np.clip(weights, min_weight, max_weight)  # 强制截断到合法范围
    weights = weights / weights.sum()  # 确保总和为1
    return pd.Series(weights, index=expected_returns.index)

def initialize(context: bigtrader.IContext):
    context.set_commission(PerOrder(buy_cost=0.0002, sell_cost=0.0002, min_cost=5))
    
    # 策略参数
    context.stock_num = hold_count  # 固定持仓4只
    context.lookback_period = 20  # 收益率计算回溯期
    context.risk_aversion = 1.0  # 风险厌恶系数
    
    if 'counter' in   context.user_store:
        pass 
    elif  'counter' not in   context.user_store:
        context.user_store['counter'] = rebalance_days - 1  # 调仓频率计数器（每5天调一次）

    context.rebalance_days = rebalance_days
 

    # 查询数据
    sql = """
    SELECT
        date,
        instrument,
        close,
        close / m_lag(close, 18) - 1 AS stop_gain_signal,
        m_avg(close, 10) / m_avg(close, 5) AS ma_ratio_5_10
    FROM cn_fund_bar1d
    """
    df = dai.query(sql, filters={"date": [context.add_trading_days(context.start_date, -350), context.end_date]}).df()
    
    # 过滤并排序数据
    context.data = df[df['instrument'].isin(specified_etfs)]
    context.data = context.data.sort_values(['instrument', 'date'])
    
    # 计算趋势评分和综合评分
    context.data['trend_score'] = context.data.groupby('instrument')['close'].transform(
        lambda x: trend_score(x, period=25)
    )
    context.data['score'] = context.data['trend_score'] + context.data['ma_ratio_5_10']
    


def before_trading_start(context, data):
    current_date = data.current_dt.strftime("%Y-%m-%d")
    context.stopwin_ins = [] 
    # print(current_date,  context.stopwin_ins)

def handle_data(context, data):
    current_date = data.current_dt.strftime("%Y-%m-%d")
    current_day_data = context.data[context.data["date"] == current_date]
    
    if len(current_day_data) == 0:
        return
    
    # 每5天调仓一次
    context.user_store['counter'] += 1
    if context.user_store['counter']  == context.rebalance_days:
        context.user_store['counter'] = 0
        current_hold = set(context.get_account_positions().keys())

        # 止盈检查：移除触发止盈的ETF
        if current_hold:
            for ins in list(current_hold):
                if ins not in current_day_data['instrument'].values:
                    context.order_target_percent(ins, 0)
                    continue
                stop_gain = current_day_data[current_day_data['instrument'] == ins]['stop_gain_signal'].values[0]
                if stop_gain > 0.16:
                    context.order_target_percent(ins, 0)
                    context.stopwin_ins.append(ins)
                    current_day_data = current_day_data[current_day_data['instrument'] != ins]

        # 筛选评分最高的4只ETF（确保有足够数量）
        if len(current_day_data) >= context.stock_num:
            current_day_data = current_day_data.sort_values('score', ascending=False).head(context.stock_num)
        else:
            # 若不足4只，直接全部保留（避免持仓不足）
            print(f"可选ETF不足{context.stock_num}只，使用全部可用ETF")
        
        stock_targets = current_day_data['instrument'].tolist()
        # 若目标列表为空或与当前持仓一致，不操作
        if not stock_targets or set(stock_targets) == current_hold:
            return

        # 卖出不在目标列表的持仓
        for ins in current_hold - set(stock_targets):
            
            if ins in context.stopwin_ins:
                continue 

            context.order_target_percent(ins, 0)

        # 计算目标ETF的权重并下单
        if stock_targets:
            try:
                # 获取历史收益数据
                data_return = context.data[context.data.date <= current_date]
                returns = calculate_returns(data_return, stock_targets, context.lookback_period)
                # 计算权重（强制4只均有正权重）
                weights = mean_variance_optimization(returns, context.risk_aversion)
                print('========:',current_date , weights) 
                # 下单
                for ins, weight in weights.items():
                    context.order_target_percent(ins, weight)
                    
            except Exception as e:
                print("优化失败，使用等权分配")
                # 异常时强制等权（确保每只都有正权重）
                equal_w = 1.0 / len(stock_targets)
                # 确保等权也满足上下限
                equal_w = np.clip(equal_w, 0.05, 0.3)
                equal_w = equal_w / (equal_w * len(stock_targets))  # 重新归一化
                for ins in stock_targets:
                    context.order_target_percent(ins, equal_w)
                    

m = bigtrader.run(
    market=bigtrader.Market.CN_FUND,
    frequency=bigtrader.Frequency.DAILY,
    start_date='2015-01-01',
    end_date='2026-03-17',
    capital_base=70000,
    initialize=initialize,
    handle_data=handle_data,
    before_trading_start=before_trading_start,
    order_price_field_buy='open',
    order_price_field_sell='open'
)

m.render() 

[2026-03-17 19:05:28] [info     ] bigtrader init ..
[2026-03-17 19:05:28] [info     ] bigtrader.run start: market=cn_fund, frequency=1d, mode=backtest, account_type=STOCK, start_date=2015-01-01, end_date=2026-03-17
[2026-03-17 19:05:28] [info     ] bigtrader<backtest> init ..
[2026-03-17 19:05:28] [info     ] prepare data ..
[2026-03-17 19:05:29] [info     ] bar1d_df: (2688616, 17)
[2026-03-17 19:05:31] [info     ] bigtrader run ..
===最后数据日期=====: 2015-01-05 00:00:00 2015-01-05 00:00:00
优化失败，使用等权分配
===最后数据日期=====: 2015-01-12 00:00:00 2015-01-12 00:00:00
优化失败，使用等权分配
===最后数据日期=====: 2015-01-19 00:00:00 2015-01-19 00:00:00
优化失败，使用等权分配
===最后数据日期=====: 2015-01-26 00:00:00 2015-01-26 00:00:00
优化失败，使用等权分配
===最后数据日期=====: 2015-02-09 00:00:00 2015-02-09 00:00:00
优化失败，使用等权分配
===最后数据日期=====: 2015-02-16 00:00:00 2015-02-16 00:00:00
优化失败，使用等权分配
===最后数据日期=====: 2015-03-16 00:00:00 2015-03-16 00:00:00
优化失败，使用等权分配
===最后数据日期=====: 2015-03-23 00:00:00 2015-03-23 00:00:00
优化失败，使用等权分配
===最后数据日期=====: 2015